# UAV Cyber-Physical Dataset Preprocessing
**Source:** IEEE DataPort — Dataset T-ITS  
**Attack Types:** DoS attack, Replay, Benign  
**Output:** X_train.npy, X_test.npy, y_train.csv, y_test.csv

## Dataset Domain Knowledge — UAV Cyber-Physical (Dataset T-ITS)

### How Drone WiFi Communication Works
Drones communicate with Ground Control Stations (GCS) over WiFi.
This dataset captures WiFi network packets between the drone and GCS.
Features are extracted from packet headers — similar to CICIDS2017
but specifically captured from real drone WiFi traffic.

### What Each Column Means

| Column | Meaning | Attack Signal |
|--------|---------|---------------|
| `frame.len` | Size of WiFi packet | Abnormal size = attack |
| `wlan.duration` | How long channel is occupied | Very high = DoS |
| `ip.ttl` | Time to live — how far packet travels | Unusual value = spoofing |
| `ip.proto` | Protocol type (TCP/UDP/etc) | Unexpected protocol = injection |
| `tcp.flags` | TCP connection flags | Unusual flags = attack probing |
| `tcp.window_size` | How much data can be sent at once | Very small = DoS effect |
| `time_since_last_packet` | Time gap between packets | Very small = flooding |
| `wlan.seq` | WiFi sequence number | Repeated numbers = Replay |

### The 3 Attack Types

**DoS attack** — overwhelm attack
- Attacker floods drone WiFi channel with traffic
- Legitimate GCS commands cannot reach the drone
- Drone loses contact with operator

**Replay** — copy-paste attack
- Attacker records legitimate GCS commands
- Replays them at wrong time or repeatedly
- Drone executes unintended commands

**Benign** — normal operation
- Regular communication between drone and GCS
- No attack present

### Key Dataset Notes
- 33,102 rows after cleaning (21,679 NaN rows removed)
- All columns were stored as strings — converted to numeric
- Relatively balanced classes — DoS≈36%, Replay≈36%, Benign≈28%
- 37 features from WiFi packet headers

### Expected XAI Findings
- **SHAP on time_since_last_packet** → key for DoS detection (very frequent packets)
- **SHAP on wlan.seq** → key for Replay detection (repeated sequence numbers)
- **LIME** → local explanation per packet, may highlight different features than SHAP
- **Permutation Importance** → confirms which features cause biggest F1 drop when shuffled
- **Key question** → do SHAP, LIME, and PI agree on top features for DoS vs Replay?

In [1]:
import os
os.makedirs("/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/raw", exist_ok=True)
os.makedirs("/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed", exist_ok=True)
print("Folders created!")

Folders created!


In [3]:
import shutil
import os

src = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical:raw/Dataset_T-ITS.csv"
dst = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/raw/Dataset_T-ITS.csv"

shutil.copy(src, dst)
print("File copied!")
print("File exists:", os.path.exists(dst))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical:raw/Dataset_T-ITS.csv'

In [4]:
import pandas as pd

df = pd.read_csv("/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/raw/Dataset_T-ITS.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel counts:")
print(df.iloc[:, -1].value_counts())

Shape: (54783, 38)
Columns: ['timestamp_c', 'frame.number', 'frame.len', 'frame.protocols', 'wlan.duration', 'wlan.ra', 'wlan.ta', 'wlan.da', 'wlan.sa', 'wlan.bssid', 'wlan.frag', 'wlan.seq', 'llc.type', 'ip.hdr_len', 'ip.len', 'ip.id', 'ip.flags', 'ip.ttl', 'ip.proto', 'ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'tcp.seq_raw', 'tcp.ack_raw', 'tcp.hdr_len', 'tcp.flags', 'tcp.window_size', 'tcp.options', 'udp.srcport', 'udp.dstport', 'udp.length', 'data.data', 'data.len', 'wlan.fc.type', 'wlan.fc.subtype', 'time_since_last_packet', 'class']

Label counts:
class
Replay        12006
DoS attack    11671
benign         9425
class             2
Name: count, dtype: int64


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_33799/2825240282.py:3: DtypeWarning: Columns (35,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/raw/Dataset_T-ITS.csv")


In [5]:
print("All unique class values:")
print(df['class'].unique())
print("\nValue counts:")
print(df['class'].value_counts())


All unique class values:
['benign' nan 'class' 'DoS attack' 'Replay']

Value counts:
class
Replay        12006
DoS attack    11671
benign         9425
class             2
Name: count, dtype: int64


In [6]:
# Check the full dataset structure
print(f"Total rows: {len(df)}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())
print(f"\nRows with NaN in class column:")
print(df[df['class'].isna()].shape[0])
print(f"\nRow 9420-9430 (around benign/DoS boundary):")
print(df.iloc[9420:9430][['class']].to_string())

Total rows: 54783

First 5 rows:
   timestamp_c frame.number frame.len frame.protocols wlan.duration wlan.ra  \
0   28105.9752           60        24               0             0       1   
1   28105.9755           61        24               0             0       1   
2  28107.09931           75       104               0             0       4   
3   28114.7857          122        86               2            44       1   
4  28114.88188          124        26               0            60       1   

  wlan.ta wlan.da wlan.sa wlan.bssid  ... tcp.options udp.srcport udp.dstport  \
0       1       1       1          0  ...           0           0           0   
1       1       1       1          0  ...           0           0           0   
2       1       4       1          0  ...           0           0           0   
3       0       1       0          0  ...           1           0           0   
4       0       1       0          0  ...           0           0           0   

  udp

In [7]:
# step 2: clean dataset  and remove NaN rows and clean duplicate header row
# NaN rows is a physical menu (sensor features), we do not need it

df_clean =df[df['class'].notna()]              # remove the class rows
df_clean = df_clean[df_clean['class']!= 'class'] # remove duplicate header

print(f"Orignial shape: {df.shape}")
print(f"Clean shape: {df_clean.shape}")
print(f"\nClass counts:")
print(df_clean['class'].value_counts())
print(f"\nMissing values per column:")
print(df_clean.isnull().sum().sum())


Orignial shape: (54783, 38)
Clean shape: (33102, 38)

Class counts:
class
Replay        12006
DoS attack    11671
benign         9425
Name: count, dtype: int64

Missing values per column:
0


In [8]:
# Cell 6: Clean dataset
df_clean = df[df['class'].notna()]
df_clean = df_clean[df_clean['class'] != 'class']

print(f"Original shape: {df.shape}")
print(f"Clean shape: {df_clean.shape}")
print(f"\nClass counts:")
print(df_clean['class'].value_counts())
print(f"\nMissing values:")
print(df_clean.isnull().sum().sum())

Original shape: (54783, 38)
Clean shape: (33102, 38)

Class counts:
class
Replay        12006
DoS attack    11671
benign         9425
Name: count, dtype: int64

Missing values:
0


In [9]:
# Cell 3: Check feature types and handle mixed columns
print("Data types:")
print(df_clean.dtypes.value_counts())

print(f"\nNon-numeric columns:")
for col in df_clean.columns:
    if df_clean[col].dtype == 'object' and col != 'class':
        print(f"  {col}: {df_clean[col].unique()[:5]}")

Data types:
object    38
Name: count, dtype: int64

Non-numeric columns:
  timestamp_c: ['28105.9752' '28105.9755' '28107.09931' '28114.7857' '28114.88188']
  frame.number: ['60' '61' '75' '122' '124']
  frame.len: ['24' '104' '86' '26' '225']
  frame.protocols: ['0' '2' '3' '1' '4']
  wlan.duration: ['0' '44' '60' '320' '162']
  wlan.ra: ['1' '4' '0' '3' '2']
  wlan.ta: ['1' '0' '2']
  wlan.da: ['1' '4' '0' '3' '2']
  wlan.sa: ['1' '0' '2']
  wlan.bssid: ['0']
  wlan.frag: ['0']
  wlan.seq: ['24' '26' '49' '14' '0']
  llc.type: ['0' '2048' '2054']
  ip.hdr_len: ['0' '20']
  ip.len: ['0' '52' '191' '192' '189']
  ip.id: ['0' '39340' '39341' '39342' '31']
  ip.flags: ['0' '64']
  ip.ttl: ['0' '128' '255' '64']
  ip.proto: ['0' '6' '17']
  ip.src: ['0' '2' '1' '3']
  ip.dst: ['0' '4' '2' '3' '1']
  tcp.srcport: ['0' '62951' '55125' '51533' '59299']
  tcp.dstport: ['0' '53']
  tcp.seq_raw: ['0' '723862127' '4045876660' '594698260' '896820386']
  tcp.ack_raw: ['0']
  tcp.hdr_len: ['0' '32'

In [10]:
# Cell 4: Convert all feature columns to numeric
# All columns are strings — need to convert to float

# Separate features and label
X = df_clean.drop(columns=['class'])
y = df_clean['class']

# Convert all feature columns to numeric
X = X.apply(pd.to_numeric, errors='coerce')

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nData types after conversion:")
print(X.dtypes.value_counts())
print(f"\nMissing values after conversion:")
print(X.isnull().sum().sum())
print(f"\nSample values:")
print(X.head(3))

X shape: (33102, 37)
y shape: (33102,)

Data types after conversion:
int64      34
float64     3
Name: count, dtype: int64

Missing values after conversion:
0

Sample values:
   timestamp_c  frame.number  frame.len  frame.protocols  wlan.duration  \
0  28105.97520            60         24                0              0   
1  28105.97550            61         24                0              0   
2  28107.09931            75        104                0              0   

   wlan.ra  wlan.ta  wlan.da  wlan.sa  wlan.bssid  ...  tcp.window_size  \
0        1        1        1        1           0  ...                0   
1        1        1        1        1           0  ...                0   
2        4        1        4        1           0  ...                0   

   tcp.options  udp.srcport  udp.dstport  udp.length  data.data  data.len  \
0            0            0            0           0          0         0   
1            0            0            0           0          0      

In [11]:
# Cell 5: Encode labels and normalize features 

from sklearn.preprocessing import LabelEncoder, StandardScaler
# step 1: encode labels
# LabelEncoder converts text labels to numbers: i.g.: 'beign' -> 0; 'DoS attack'->1
# reason: ML need number instead of text.

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Classes: {list(le.classes_)}")
print(f"Encoded as: {list(range(len(le.classes_)))}")
print(f"y_encorded shape: {y_encoded.shape}")

# step 2:Normalize features
# standardScaler converts all features to same scale: formula: (value- mean) / std
# reason: prevents large numbers dominating small numbers
# i.g.: port number 8889 vs flag value 0 or 1
# after scaling: all features have mean = 0, std = 1

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nX_scaled shape: {X_scaled.shape}")
print(f"Mean after scaling (should be ~0): {X_scaled.mean().round(4)}")
print(f"Std after scaling (should be ~1): {X_scaled.std().round(4)}")
print("\nEncoding and scaling complete!")

# y_encoded shape: (33102,) -> y_encoded = [0, 2, 1, 0, 2, ...]  # 33,102 numbers
# Each number(1D array ) = one class label for one row

# X_scaled shape: (33102, 37) -> number of row and number column

Classes: ['DoS attack', 'Replay', 'benign']
Encoded as: [0, 1, 2]
y_encorded shape: (33102,)

X_scaled shape: (33102, 37)
Mean after scaling (should be ~0): -0.0
Std after scaling (should be ~1): 0.9586

Encoding and scaling complete!


In [12]:
# Cell 6: Split data and save to processed folder
from sklearn.model_selection import train_test_split

# Split 70/30 same as CICIDS2017
# reason: 70% for training, 30% for testing
# random_state=42 ensures same split every run (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded  # keeps same class ratio in train/test
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

print(f"\nTrain class distribution:")
import numpy as np
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.classes_[u]}: {c}")

print(f"\nTest class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.classes_[u]}: {c}")

X_train shape: (23171, 37)
X_test shape:  (9931, 37)
y_train shape: (23171,)
y_test shape:  (9931,)

Train class distribution:
  DoS attack: 8170
  Replay: 8404
  benign: 6597

Test class distribution:
  DoS attack: 3501
  Replay: 3602
  benign: 2828


In [13]:
# Cell 7: save processed files to processed folder

import numpy as np 
import pandas as pd

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"

# save numpy arrays (for CNN and Autoencoder)
# .npy files load faster than CSV for large arrays 
np.save(save_path + "X_train.npy", X_train)
np.save(save_path + "X_test.npy", X_test)

# save labels as CSV (for RF and evaluation)
# CSV keeps class names readble
pd.DataFrame(y_train).to_csv(save_path + "y_train.csv", index=False)
pd.DataFrame(y_test).to_csv(save_path + "y_test.csv", index=False)

# save feature names (for SHAP and LIME)
# XAI methods need feature names for plots
feature_names = list(X.columns)
pd.DataFrame(feature_names).to_csv(save_path + "feature_names.csv",index=False)

# Save label encoder classes
# needed to convert numbers back to class names
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("Saved files:")
print(f" - X_train.npy  {X_train.shape}")
print(f" - X_test.npy  {X_test.shape}")
print(f" - y_train.csv  {y_train.shape}")
print(f" - y_test.csv {y_test.shape}")
print(f" - feature_names.csv ({len(feature_names)} features)")
print(f" - label_classes.csv ({len(le.classes_)} classes)")
print("\nPreprocessing 100% complete!")


      

Saved files:
 - X_train.npy  (23171, 37)
 - X_test.npy  (9931, 37)
 - y_train.csv  (23171,)
 - y_test.csv (9931,)
 - feature_names.csv (37 features)
 - label_classes.csv (3 classes)

Preprocessing 100% complete!


In [16]:
# save feature names and label classes — same as new datasets
# goal: all 5 datasets have identical file structure for universal loading
pd.DataFrame(X.columns).to_csv(save_path + "feature_names.csv", index=False)
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("feature_names.csv saved")
print("label_classes.csv saved")

feature_names.csv saved
label_classes.csv saved


## Preprocessing Complete

| Item | Value |
|------|-------|
| Source | Dataset T-ITS — UAV WiFi Communication |
| Original rows | 54,783 |
| After cleaning | 33,102 (removed 21,681 NaN + 2 duplicate header rows) |
| Total features | 37 |
| Labels | DoS attack=0, Replay=1, benign=2 |
| Class balance | Relatively balanced — DoS 36%, Replay 36%, Benign 28% |
| Train set | 23,171 rows |
| Test set | 9,931 rows |
| Split ratio | 70% train / 30% test |
| Output files | X_train.npy, X_test.npy, y_train.csv, y_test.csv |